# AutoGPT

https://github.com/Significant-Gravitas/Auto-GPT 的实现，但使用了 LangChain 的基础组件（LLMs、PromptTemplates、VectorStores、Embeddings、Tools）

## 设置工具

我们将设置一个 AutoGPT，包含一个搜索工具、一个写文件工具和一个读文件工具。

In [2]:
from langchain.agents import Tool
from langchain_community.tools.file_management.read import ReadFileTool
from langchain_community.tools.file_management.write import WriteFileTool
from langchain_community.utilities import SerpAPIWrapper

search = SerpAPIWrapper()
tools = [
    Tool(
        name="search",
        func=search.run,
        description="useful for when you need to answer questions about current events. You should ask targeted questions",
    ),
    WriteFileTool(),
    ReadFileTool(),
]

## 设置内存

这里的内存用于代理的中间步骤

In [3]:
from langchain.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

In [4]:
# Define your embedding model
embeddings_model = OpenAIEmbeddings()
# Initialize the vectorstore as empty
import faiss

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})

## 设置模型和 AutoGPT

初始化一切！我们将使用 ChatOpenAI 模型

In [5]:
from langchain_experimental.autonomous_agents import AutoGPT
from langchain_openai import ChatOpenAI

In [6]:
agent = AutoGPT.from_llm_and_tools(
    ai_name="Tom",
    ai_role="Assistant",
    tools=tools,
    llm=ChatOpenAI(temperature=0),
    memory=vectorstore.as_retriever(),
)
# Set verbose to be true
agent.chain.verbose = True

## 运行示例

在这里，我们将让它撰写一份旧金山的天气预报。

In [ ]:
agent.run(["write a weather report for SF today"])

## 聊天记录记忆

除了存储代理即时步骤的记忆外，我们还有一个聊天记录记忆。默认情况下，代理将使用“ChatMessageHistory”，并且可以更改。当您想使用不同类型的记忆时，例如“FileChatHistoryMemory”，这将非常有用。

In [ ]:
from langchain_community.chat_message_histories import FileChatMessageHistory

agent = AutoGPT.from_llm_and_tools(
    ai_name="Tom",
    ai_role="Assistant",
    tools=tools,
    llm=ChatOpenAI(temperature=0),
    memory=vectorstore.as_retriever(),
    chat_history_memory=FileChatMessageHistory("chat_history.txt"),
)